### **Top 10 Anomalous Positive Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'anomalous_positive'
ORDER BY p.total_revenue DESC
LIMIT 10

### **Top 10 Inelastic Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'inelastic'
ORDER BY p.total_revenue DESC
LIMIT 10

### **Top 10 Elastic Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'elastic'
ORDER BY p.total_revenue DESC
LIMIT 10

### **Top 15 products that contributed highest revenue overall**

In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
),

joined AS (
    SELECT
        e.Description,
        e.elasticity,
        e.category,
        r.total_revenue
    FROM elasticity_table e
    JOIN revenue r
    ON e.Description = r.Description
),

ranked AS (
    SELECT
        *,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        SUM(total_revenue) OVER () AS total_revenue_all,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue
    FROM joined
)

SELECT
    Description,
    elasticity,
    category,
    total_revenue,
    revenue_rank,
    cumulative_revenue,
    (cumulative_revenue / total_revenue_all) * 100 AS cumulative_pct
FROM ranked
ORDER BY revenue_rank
limit (15)


## **The Pricing strategy table**

### Below is the final business aggregate table that marks goods with different elasticity threshold to be categorized as whether to INCREASE VOLUME or PRICE or weather they are LUXUARY goods with always PREMIMUM PRICING and lastly goods that are to be TESTED.

### The table includes high revenue products first along with their elasticity value.

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
),

thresholds AS (
    SELECT
        percentile_approx(total_revenue, 0.75) AS rev_p75,
        percentile_approx(total_qty, 0.75) AS qty_p75
    FROM product_stats
)

SELECT
    e.Description,
    e.category,
    e.elasticity,
    p.total_revenue,
    p.avg_price,
    p.total_qty,

    CASE
        -- Strong inelastic + high demand → safe price increase
        WHEN e.category = 'inelastic'
             AND ABS(e.elasticity) < 0.5
             AND p.total_qty >= t.qty_p75
        THEN 'Increase Price (Safe)'

        -- Elastic + good demand → volume strategy
        WHEN e.category = 'elastic'
             AND ABS(e.elasticity) > 1
             AND p.total_qty >= t.qty_p75
        THEN 'Increase Volume'

        -- Anomalous + high revenue → premium pricing
        WHEN e.category = 'anomalous_positive'
             AND p.total_revenue >= t.rev_p75
        THEN 'Premium Pricing'

        -- Others → uncertain
        ELSE 'Monitor / Test'
    END AS pricing_strategy

FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
CROSS JOIN thresholds t
ORDER BY p.total_revenue DESC